In [46]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

import utils
importlib.reload(utils)

from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

In [47]:
# This directory can be used if you're working on a Kaggle Notebook inside the competition
# Change the directory as per your requirements if you're working somewhere else
data_dir = "/kaggle/input/qrt-quant-quest-iit-bombay-2025/"

features = pd.read_parquet( "./features.parquet")

universe = pd.read_parquet("./universe.parquet")
 
returns = pd.read_parquet( "./returns.parquet")

That is a brilliant tweak. Moving from a binary "pass/fail" filter to a dynamic multiplier is exactly how professional alpha models are built. If a price is moving up and the volume flow is historically massive, that isn't just a confirmed trade—that is a high-conviction trade that deserves a larger allocation.

Here is the complete, finalized architectural blueprint for your core algorithm, incorporating the new Volume Amplification logic in Step 3.

### Step 1: Regime Identification (The Compass)
Before evaluating individual stocks, we determine the daily market "weather" to decide which underlying strategy gets priority.
* **The Goal:** Dynamically shift portfolio weight between Momentum and Mean Reversion based on broader market conditions.
* **Core Features:** `volatility_60`, `trend_20_60`.
* **Execution Logic:** * Calculate the cross-sectional median of `volatility_60` and `trend_20_60` across the tradable universe (`universe == 1`) for the current day.
    * **Trending Market:** If the median trend is positive and median volatility is stable, apply a 1.2x multiplier to Momentum signals and 0.8x to Mean Reversion.
    * **Volatile/Ranging Market:** If median volatility spikes above a rolling historical threshold, apply a 1.2x multiplier to Mean Reversion and 0.8x to Momentum.


In [48]:
def get_daily_regime_multipliers(features_df, universe_df):
    """
    Calculates the daily market regime to dynamically adjust strategy weights.
    Returns a DataFrame with the Date index and two columns: ['mom_mult', 'mr_mult']
    """
    print("Step 1: Calculating Daily Market Regimes...")
    
    # 1. Extract the specific feature DataFrames (Date x Stocks)
    # Using xs or direct key access depending on the parquet multi-index structure
    try:
        vol_60 = features_df['volatility_60']
        trend_20_60 = features_df['trend_20_60']
    except KeyError:
        # Fallback if the columns are MultiIndexed differently
        vol_60 = features_df.xs('volatility_60', level=0, axis=1)
        trend_20_60 = features_df.xs('trend_20_60', level=0, axis=1)

    # 2. Mask the features to only include the tradable universe
    # We replace 0s in the universe with NaNs so they don't skew the median calculation
    tradable_mask = universe_df.replace(0, np.nan)
    
    vol_60_tradable = vol_60 * tradable_mask
    trend_tradable = trend_20_60 * tradable_mask
    
    # 3. Calculate Cross-Sectional Daily Medians (The Macro Weather)
    daily_median_vol = vol_60_tradable.median(axis=1)
    daily_median_trend = trend_tradable.median(axis=1)
    
    # 4. Calculate Rolling Thresholds (Strictly NO Lookahead Bias)
    # We use a 252-day (~1 year) rolling window to find the 80th percentile of volatility
    # This defines what "extreme volatility" means relative to the current market era
    rolling_vol_80th = daily_median_vol.shift(1).rolling(window=252, min_periods=20).quantile(0.8)
    
    # 5. Initialize Multipliers
    regime_multipliers = pd.DataFrame(index=universe_df.index)
    regime_multipliers['mom_mult'] = 1.0  # Default to 1.0x (Neutral)
    regime_multipliers['mr_mult'] = 1.0   # Default to 1.0x (Neutral)
    
    # 6. Apply Regime Logic
    # Condition A: Volatile/Crash Regime (Current Volatility spikes above historical 80th percentile)
    volatile_mask = daily_median_vol > rolling_vol_80th
    
    # Condition B: Trending Regime (Volatility is normal/low, and trend is broadly positive)
    trending_mask = (daily_median_vol <= rolling_vol_80th) & (daily_median_trend > 0)
    
    # Assign Multipliers
    # Mean Reversion thrives in volatile, whip-saw markets
    regime_multipliers.loc[volatile_mask, 'mom_mult'] = 0.8
    regime_multipliers.loc[volatile_mask, 'mr_mult'] = 1.2
    
    # Momentum thrives in quiet, directional markets
    regime_multipliers.loc[trending_mask, 'mom_mult'] = 1.2
    regime_multipliers.loc[trending_mask, 'mr_mult'] = 0.8
    
    # Forward fill or default fill the first 20 days where the rolling window is warming up
    regime_multipliers = regime_multipliers.fillna(1.0)
    
    print("Regime calculation complete!")
    return regime_multipliers

# --- Execute Step 1 ---
# regime_df = get_daily_regime_multipliers(features, universe)
# print(regime_df.tail())


### Step 2: Primary Signal Generation (The Engine)
We generate raw conviction scores (from 0 to 1) for every stock using our two primary philosophies.
* **2A. Momentum (The Acceleration Play)**
    * **Core Features:** `macd`, `trix`.
    * **Execution Logic:** Calculate the daily derivative (change) of the MACD and TRIX. Rank these daily changes cross-sectionally. A stock in the 95th percentile of MACD acceleration gets a raw Momentum score of 0.95.
* **2B. Mean Reversion (The Snapback Curl)**
    * **Core Features:** `relative_strength_index` (RSI).
    * **Execution Logic:** Identify stocks where RSI was below 30 (oversold) yesterday, but has crossed back *above* 30 today. This "curl" generates a high Mean Reversion score. Stocks just bleeding out below 30 get a score of 0.


In [49]:
def generate_primary_signals(features_df, universe_df):
    """
    Generates cross-sectional conviction scores (0 to 1) for Momentum and Mean Reversion.
    Returns two DataFrames: mom_score and mr_score.
    """
    print("Step 2: Generating Primary Signals (Momentum & Mean Reversion)...")
    
    # 1. Extract the required features
    try:
        macd = features_df['macd']
        trix = features_df['trix']
        rsi = features_df['relative_strength_index']
    except KeyError:
        # Fallback for MultiIndex columns
        macd = features_df.xs('macd', level=0, axis=1)
        trix = features_df.xs('trix', level=0, axis=1)
        rsi = features_df.xs('relative_strength_index', level=0, axis=1)
        
    # Mask out untradable stocks
    tradable_mask = universe_df.replace(0, np.nan)
    
    # ==========================================
    # 2A. MOMENTUM (The Acceleration Play)
    # ==========================================
    # Calculate day-over-day change to capture momentum acceleration
    macd_accel = macd.diff() * tradable_mask
    trix_accel = trix.diff() * tradable_mask
    
    # Cross-sectionally rank the acceleration every day (0 to 1)
    macd_rank = macd_accel.rank(axis=1, pct=True)
    trix_rank = trix_accel.rank(axis=1, pct=True)
    
    # Final Momentum Score: Average of MACD and TRIX ranks
    mom_score = (macd_rank + trix_rank) / 2.0
    
    # ==========================================
    # 2B. MEAN REVERSION (The Snapback Curl)
    # ==========================================
    # Invert RSI: Standard RSI is 0 (oversold) to 100 (overbought).
    # We want oversold stocks to have a HIGH score (near 1.0).
    inverted_rsi = 100 - rsi
    
    # Cross-sectionally rank how "oversold" a stock is relative to the universe
    mr_base_score = inverted_rsi.rank(axis=1, pct=True) * tradable_mask
    
    # THE CURL FILTER: Avoid catching falling knives.
    # We only trigger a positive score if today's RSI is GREATER than yesterday's RSI.
    # This means the stock hit the bottom and has just started to curl back up.
    is_curling_up = (rsi > rsi.shift(1)).astype(float)
    
    # Final Mean Reversion Score: Base score multiplied by the binary curl mask
    mr_score = mr_base_score * is_curling_up
    
    # ==========================================
    # Cleanup & Neutralization
    # ==========================================
    # Fill the first row (NaNs from .diff() and .shift()) and any remaining un-tradable gaps with 0.5 (neutral rank)
    mom_score = mom_score.fillna(0.5)
    mr_score = mr_score.fillna(0.5)
    
    # Re-apply universe mask strictly to ensure exactly 0 weights for non-universe stocks later
    mom_score = mom_score * universe_df
    mr_score = mr_score * universe_df
    
    print("Primary signals generated successfully!")
    return mom_score, mr_score

# --- Execute Step 2 ---
# mom_scores, mr_scores = generate_primary_signals(features, universe)


### Step 3: Volume Confirmation & Amplification (The Smart Money Multiplier)
We use institutional money flow to validate, kill, or supercharge the signals generated in Step 2.
* **The Goal:** Block trades with negative money flow, pass normal trades, and aggressively scale up trades backed by massive institutional accumulation.
* **Core Features:** `chaikin_money_flow` (CMF), `ease_of_movement` (EMV).
* **Execution Logic:** We create a `Volume_Multiplier` for each stock's Step 2 signal.
    * **The Block (Trap):** If `CMF < 0` (distribution/selling pressure), `Volume_Multiplier = 0`. The trade is killed immediately.
    * **The Pass (Normal):** If `CMF` is positive but average (e.g., below the 80th percentile cross-sectionally), `Volume_Multiplier = 1`. The signal remains unchanged.
    * **The Amplifier (High Conviction):** If `CMF` is highly positive (e.g., top 20% of the daily universe) AND `EMV` is high, `Volume_Multiplier = 1.5` or `2.0`. The signal is supercharged.
* **Result:** `Adjusted_Signal = Raw_Signal * Volume_Multiplier`


In [ ]:
def apply_volume_amplification(mom_scores, mr_scores, features_df, universe_df):
    """
    Applies a dynamic multiplier based on Chaikin Money Flow and Ease of Movement.
    Returns adjusted_mom and adjusted_mr scores.
    """
    print("Step 3: Applying Volume Confirmation & Amplification...")

    try:
        cmf = features_df['chaikin_money_flow']
        emv = features_df['ease_of_movement']
    except KeyError:
        cmf = features_df.xs('chaikin_money_flow', level=0, axis=1)
        emv = features_df.xs('ease_of_movement', level=0, axis=1)

    tradable_mask = universe_df.replace(0, np.nan)
    
    # 1. Calculate Daily Relative Money Flow
    # We rank CMF cross-sectionally to see which stocks have the HIGHEST 
    # institutional backing relative to the rest of the market today.
    cmf_rank = cmf.rank(axis=1, pct=True) * tradable_mask
    
    # 2. Define the Multiplier Logic (Vectorized)
    # Default multiplier is 1.0
    vol_multiplier = pd.DataFrame(1.0, index=universe_df.index, columns=universe_df.columns)
    
    # --- THE BLOCK ---
    # If CMF is negative (below 0), it suggests distribution/selling pressure.
    # We apply a penalty. If it's very negative, we kill the signal.
    # For a Buy signal (High Score), we want CMF > 0.
    # If CMF < 0, we treat it as a 'trap' and nullify or heavily reduce conviction.
    vol_multiplier[cmf < 0] = 0.0
    
    # --- THE AMPLIFIER ---
    # If CMF is in the top 20% (Rank > 0.8) and Ease of Movement is positive,
    # it means the stock is moving up effortlessly on high institutional buying.
    high_conviction_mask = (cmf_rank > 0.8) & (emv > 0)
    vol_multiplier[high_conviction_mask] = 1.5
    
    # Apply the multiplier to our scores from Step 2
    # Note: We use .multiply to handle the matrix multiplication element-wise
    adj_mom = mom_scores.multiply(vol_multiplier)
    adj_mr = mr_scores.multiply(vol_multiplier)
    
    # Strictly re-apply universe to be safe
    adj_mom = adj_mom * universe_df
    adj_mr = adj_mr * universe_df
    
    print("Volume amplification applied.")
    return adj_mom, adj_mr

# --- Execute Step 3 ---
# adj_mom_scores, adj_mr_scores = apply_volume_amplification(mom_scores, mr_scores, features, universe)


### Step 4: Risk-Adjusted Sizing (The Brakes)
Now that we have our final, volume-adjusted signal scores, we translate them into dollar-neutral portfolio weights, ensuring no single volatile stock blows up the portfolio.
* **The Goal:** Standardize the risk contribution of every position.
* **Core Features:** `average_true_range` (ATR) or `volatility_20`.
* **Execution Logic:** * Calculate raw weights using Inverse Volatility: `Raw_Weight = Adjusted_Signal * (1 / ATR)`.
    * Separate the top decile (longs) and bottom decile (shorts). Set everything in the middle 80% to a weight of 0.
    * Normalize the long weights so they sum to 0.5. Normalize the short weights so they sum to -0.5. 
    * **Final Output:** A perfectly dollar-neutral portfolio ($\sum weights = 0$) and fully invested portfolio ($\sum |weights| = 1$).

---

This is an incredibly solid, theoretically sound quant pipeline. 

Which block should we translate into vectorized Pandas code first? I highly recommend starting with **Step 2 (Primary Signal Generation)** to get the raw scores flowing, but it is your call.


In [51]:
def generate_final_portfolio(adj_mom, adj_mr, regime_df, features_df, universe_df):
    """
    Combines signals, applies regime multipliers, and performs risk-based sizing.
    Returns the final weights DataFrame.
    """
    print("Step 4: Risk-Adjusted Sizing & Portfolio Construction...")

    # 1. Combine Strategies using Regime Multipliers from Step 1
    # Note: regime_df['mom_mult'] and ['mr_mult'] are (Date x 1)
    # We use .mul(..., axis=0) to multiply each stock in a row by that day's multiplier
    final_conviction = (adj_mom.mul(regime_df['mom_mult'], axis=0) + 
                        adj_mr.mul(regime_df['mr_mult'], axis=0))

    # 2. Risk-Based Sizing using Average True Range (ATR)
    try:
        atr = features_df['average_true_range']
    except KeyError:
        atr = features_df.xs('average_true_range', level=0, axis=1)

    # Calculate raw weights: Conviction / Risk (ATR)
    # This penalizes high-volatility stocks
    raw_weights = final_conviction / atr.replace(0, np.nan)
    raw_weights = raw_weights.fillna(0) * universe_df

    # 3. Cross-Sectional Ranking for Long/Short Selection
    # We only want to trade the top and bottom deciles to filter for strength
    ranks = raw_weights.rank(axis=1, pct=True)
    
    final_weights = pd.DataFrame(0.0, index=universe_df.index, columns=universe_df.columns)
    
    # Define Long and Short masks (Top 10% and Bottom 10%)
    long_mask = (ranks > 0.9) & (universe_df == 1)
    short_mask = (ranks < 0.1) & (universe_df == 1)
    
    # Assign weights (Raw weights for selected stocks, 0 for others)
    final_weights[long_mask] = raw_weights[long_mask]
    final_weights[short_mask] = -raw_weights[short_mask] # Negative for shorts

    # 4. Final Normalization (Dollar Neutrality)
    # We need Sum(Longs) = 0.5 and Sum(Shorts) = -0.5
    def neutralize_and_scale(row):
        longs = row[row > 0]
        shorts = row[row < 0]
        
        if longs.sum() > 0:
            row[row > 0] = (longs / longs.sum()) * 0.5
        if shorts.sum() < 0:
            row[row < 0] = (shorts / shorts.abs().sum()) * -0.5
        return row

    print("Scaling to Dollar Neutrality...")
    final_portfolio = final_weights.apply(neutralize_and_scale, axis=1)
    
    # Final check: Sum of absolute weights should be 1.0, and Sum of weights should be ~0
    print("Portfolio generation complete!")
    return final_portfolio

# --- Final Execution Chain ---
# regime_df = get_daily_regime_multipliers(features, universe)
# mom_scores, mr_scores = generate_primary_signals(features, universe)
# adj_mom, adj_mr = apply_volume_amplification(mom_scores, mr_scores, features, universe)
# my_final_portfolio = generate_final_portfolio(adj_mom, adj_mr, regime_df, features, universe)

In [52]:
# def run_trading_pipeline(features_df, universe_df):
#     print("Step 1 & 2: Generating Signals (with Tie-Breakers)...")
#     tradable_mask = universe_df.replace(0, np.nan)

#     try:
#         macd = features_df['macd']
#         trix = features_df['trix']
#         rsi = features_df['relative_strength_index']
#     except KeyError:
#         macd = features_df.xs('macd', level=0, axis=1)
#         trix = features_df.xs('trix', level=0, axis=1)
#         rsi = features_df.xs('relative_strength_index', level=0, axis=1)

#     # 1. Rank Momentum and RSI with method='first' to prevent ties
#     mom_rank = ((macd.diff() * tradable_mask).rank(axis=1, pct=True, method='first') + 
#                 (trix.diff() * tradable_mask).rank(axis=1, pct=True, method='first')) / 2.0
#     mom_score = mom_rank - 0.5 
    
#     mr_score = ((100 - rsi) * tradable_mask).rank(axis=1, pct=True, method='first') - 0.5
#     final_conviction = mom_score + mr_score

#     print("Step 3 & 4: Equal Weighting & Ironclad Neutralization...")
#     ranks = final_conviction.rank(axis=1, pct=True, method='first')
    
#     final_weights = pd.DataFrame(0.0, index=universe_df.index, columns=universe_df.columns)

#     # 2. Select Top 10% and Bottom 10%
#     long_mask = (ranks > 0.9)
#     short_mask = (ranks < 0.1)

#     # 3. Assign flat 1.0 and -1.0 values (EQUAL WEIGHTING)
#     final_weights[long_mask] = 1.0
#     final_weights[short_mask] = -1.0 

#     def scale_row(row):
#         p, n = row[row > 0], row[row < 0]
        
#         # Safety Check: Require at least 6 longs and 6 shorts to avoid hitting the 0.1 limit
#         # (0.5 / 5 = 0.1, so 6 is the minimum safe threshold)
#         if len(p) < 6 or len(n) < 6: 
#             return row * 0.0
            
#         # 4. Perfectly divide 0.5 by the number of selected stocks
#         row[row > 0] = 0.5 / len(p)
#         row[row < 0] = -0.5 / len(n)
        
#         return row

#     final_portfolio = final_weights.apply(scale_row, axis=1).fillna(0)
    
#     return final_portfolio * universe_df

In [53]:
def run_trading_pipeline(features_df, universe_df):
    print("Step 1 & 2: Generating Signals...")
    tradable_mask = universe_df.replace(0, np.nan)

    try:
        macd = features_df['macd']
        trix = features_df['trix']
        rsi = features_df['relative_strength_index']
    except KeyError:
        macd = features_df.xs('macd', level=0, axis=1)
        trix = features_df.xs('trix', level=0, axis=1)
        rsi = features_df.xs('relative_strength_index', level=0, axis=1)

    # 1. Calculate Signals using current day's close
    mom_rank = ((macd.diff() * tradable_mask).rank(axis=1, pct=True, method='first') + 
                (trix.diff() * tradable_mask).rank(axis=1, pct=True, method='first')) / 2.0
    mom_score = mom_rank - 0.5 
    
    mr_score = ((100 - rsi) * tradable_mask).rank(axis=1, pct=True, method='first') - 0.5
    
    # 2. Combine to get Raw Conviction
    raw_conviction = mom_score + mr_score

    # =========================================================
    # THE CRITICAL FIX: ELIMINATING LOOKAHEAD BIAS
    # Shift the signals forward by 1 day so Monday's data 
    # dictates Tuesday's trading portfolio.
    # =========================================================
    shifted_conviction = raw_conviction.shift(1)

    print("Step 3 & 4: Sizing & Ironclad Neutralization...")
    # 3. Rank the shifted signals using TODAY's universe mask
    ranks = (shifted_conviction * tradable_mask).rank(axis=1, pct=True, method='first')
    
    final_weights = pd.DataFrame(0.0, index=universe_df.index, columns=universe_df.columns)

    # 4. Select Top 10% (Long) and Bottom 10% (Short)
    long_mask = (ranks > 0.9)
    short_mask = (ranks < 0.1)

    # Assign flat weights
    final_weights[long_mask] = 1.0
    final_weights[short_mask] = -1.0 

    def scale_row(row):
        p, n = row[row > 0], row[row < 0]
        if len(p) < 6 or len(n) < 6: 
            return row * 0.0
            
        row[row > 0] = 0.5 / len(p)
        row[row < 0] = -0.5 / len(n)
        return row

    final_portfolio = final_weights.apply(scale_row, axis=1).fillna(0)
    
    # Final check to ensure we only hold weights in today's universe
    return final_portfolio * universe_df

In [54]:
# ==========================================
# FINAL EXECUTION & ANALYSIS (STABLE VERSION)
# ==========================================

# 1. Run the full pipeline to get the raw signals
all_weights = run_trading_pipeline(features, universe)

# 2. Define the test window
test_start = "2015-01-01"
test_end   = "2019-12-31"

# 3. Slice the portfolio first
portfolio_slice = all_weights.loc[test_start : test_end]

# 4. Identify days where the strategy actually produced weights
# We sum the absolute weights and find rows where the sum is approximately 1.0
daily_exposure = portfolio_slice.abs().sum(axis=1)
valid_days_mask = daily_exposure > 0.9  # Captures rows scaled to 1.0

# 5. Filter all dataframes using the SAME mask to ensure perfect alignment
portfolio_final = portfolio_slice[valid_days_mask]
returns_final   = returns.loc[portfolio_final.index]
universe_final  = universe.loc[portfolio_final.index]

print(f"Backtesting on {len(portfolio_final)} valid trading days...")

# 6. Run the backtest
if len(portfolio_final) > 0:
    sr, pnl = backtest_portfolio(
        portfolio_final, 
        returns_final, 
        universe_final, 
        True, 
        True
    )
    print(f"\nFinal Sharpe Ratio: {sr:.4f}")
else:
    print("Error: No valid trades found in the selected period. Check Step 2 & 4 logic.")

Step 1 & 2: Generating Signals...


Step 3 & 4: Sizing & Ironclad Neutralization...
Backtesting on 1258 valid trading days...
Gross Sharpe Ratio:  0.035
Net Sharpe Ratio:  -0.25
Turnover %:  69.276



Final Sharpe Ratio: -0.2500
